# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Same filter as w04 baseline — real demand only
df = df[df["impressions_90d"] >= 100].copy()

print("Rows:", len(df))
df.head(3)

Rows: 22006


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. Method choice and why

I'm using **Logistic Regression** as a simple, interpretable first model, then **Random Forest** as 
a stronger method — the same two-step comparison the starter pipeline itself uses, where Random 
Forest clearly beat both the baseline rule and logistic regression (Precision@50: baseline 0.240 → 
logistic 0.400 → random forest 0.740).

**Why these fit my lane:** my task is classification (`is_declining_label`) turned into a ranking 
(top-K review queue) — exactly what these two methods are built for. Logistic Regression gives me an 
honest, explainable starting point (coefficients I can read directly). Random Forest can capture 
non-obvious combinations of signals (e.g. position × CTR × impressions interacting) that a linear 
model or my hand-written baseline rule can't — which is exactly the kind of "worth the complexity" 
justification I need, since the assignment penalizes complexity that doesn't earn its keep.

I'm not using clustering here since my lane is about ranking pages by priority, not grouping them 
into unlabeled archetypes (that's Lane 3's job).

In [14]:
# Target: same proxy label used throughout (from w01-w02 framing)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Declining pages:", df["is_declining_label"].sum())
print("Total pages:", len(df))
print("Share declining:", round(df["is_declining_label"].mean() * 100, 1), "%")

# Feature columns — same signals used in w04 baseline (no leakage)
feature_cols = [
    "avg_position", "ctr", "impressions_90d", "sessions_90d",
    "word_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition"
]

X = df[feature_cols].fillna(0)
y = df["is_declining_label"]

print("\nFeature columns:", feature_cols)
print("X shape:", X.shape)

Declining pages: 13152
Total pages: 22006
Share declining: 59.8 %

Feature columns: ['avg_position', 'ctr', 'impressions_90d', 'sessions_90d', 'word_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition']
X shape: (22006, 9)


In [15]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), " | Test rows:", len(X_test))
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

# Confirm no client overlap
overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print("Overlapping clients (should be 0):", len(overlap))

Train rows: 18392  | Test rows: 3614
Train clients: 24
Test clients: 6
Overlapping clients (should be 0): 0


## 2. Split design

**I'm using a client-grouped split** (80% train, 20% test by client, not by row).

**Why this is honest for my question:** pages from the same client often share patterns — similar 
content strategy, similar SEO practices, similar baseline traffic levels. If I split by row 
randomly, the model could see some pages from a client in training and other pages from that same 
client in testing, letting it partly "memorize" client-specific quirks rather than learning signals 
that generalize to genuinely new clients. Since the real-world use case is scoring pages for clients 
(including ones the model hasn't necessarily learned every quirk of), a client-grouped split gives a 
more honest estimate of how the model will perform in practice.

I verified there's zero client overlap between train and test after the split (printed above).

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Logistic Regression (scaled features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_proba = log_model.predict_proba(X_test_scaled)[:, 1]

# Random Forest (no scaling needed)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("Models trained.")

Models trained.


In [17]:
def precision_at_k(y_true, scores, k=50):
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].sum() / k

# Model scores
log_p50 = precision_at_k(y_test, log_proba, k=50)
rf_p50 = precision_at_k(y_test, rf_proba, k=50)

# Baseline: recompute the w04 opportunity_score on the SAME test set for fair comparison
test_df = df.iloc[test_idx].copy()

position_bucket_map = pd.cut(
    test_df["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1_top3", "2_top10", "3_top20", "4_below20"]
)
expected_ctr_lookup = test_df.groupby(position_bucket_map)["ctr"].transform("mean")
test_df["ctr_gap"] = (expected_ctr_lookup - test_df["ctr"]).clip(lower=0)
test_df["impressions_norm"] = (
    (test_df["impressions_90d"] - test_df["impressions_90d"].min()) /
    (test_df["impressions_90d"].max() - test_df["impressions_90d"].min())
)
test_df["ctr_gap_norm"] = (
    (test_df["ctr_gap"] - test_df["ctr_gap"].min()) /
    (test_df["ctr_gap"].max() - test_df["ctr_gap"].min())
)
test_df["baseline_score"] = 0.6 * test_df["ctr_gap_norm"] + 0.4 * test_df["impressions_norm"]

baseline_p50 = precision_at_k(y_test.reset_index(drop=True), test_df["baseline_score"].reset_index(drop=True), k=50)

comparison = pd.DataFrame({
    "Method": ["Baseline rule (w04)", "Logistic Regression", "Random Forest"],
    "Precision@50": [baseline_p50, log_p50, rf_p50]
})
comparison

,Method,Precision@50
0,Baseline rule (w04),0.82
1,Logistic Regression,0.88
2,Random Forest,0.54


## 3. Train + compare vs my baseline

| Method | Precision@50 |
|---|---:|
| Baseline rule (w04) | 0.82 |
| Logistic Regression | 0.88 |
| Random Forest | 0.54 |

**Result:** Logistic Regression is my best method here (0.88), slightly beating my own w04 baseline 
rule (0.82). Random Forest performed noticeably worse (0.54) — this is the honest number, not the 
result I expected going in (the starter pipeline's own numbers showed Random Forest winning by a 
wide margin, so I want to be clear I'm not cherry-picking a result that fits a script).

**Why I think Random Forest underperformed here:**
- My test set only has 6 clients (3,614 rows) — Random Forest with 200 trees and max_depth=8 may be 
  overfitting to patterns in the 24 training clients that don't generalize to this particular small, 
  different set of test clients.
- My baseline rule and Logistic Regression are both essentially linear/monotonic in how they use 
  `avg_position` and `ctr` — and my label (`trend_direction == "down"`) may itself correlate close 
  to linearly with these signals, so a simpler model naturally fits better here than a model that 
  hunts for more complex interactions that may not actually exist in this label.
- This matches the assignment's own warning: complexity should be rewarded only if it earns its 
  keep. Here, it didn't — so I'm keeping Logistic Regression as my chosen model, not Random Forest, 
  even though Random Forest is the "fancier" option.

In [18]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

coef_df

,feature,coefficient
5,content_age_days,-0.356600
1,ctr,-0.324227
0,avg_position,-0.314135
4,word_count,0.226680
3,sessions_90d,-0.149151
2,impressions_90d,-0.120302
6,days_since_last_update,0.093792
8,competition,0.038635
7,search_volume,0.006940


## 4. Errors and interpretation

**What the model leans on most** (Logistic Regression coefficients, sorted by strength):

| Feature | Coefficient | Direction |
|---|---:|---|
| `content_age_days` | -0.357 | older pages → *less* likely flagged declining |
| `ctr` | -0.324 | higher CTR → less likely declining |
| `avg_position` | -0.314 | lower (better) position number → less likely declining |
| `word_count` | +0.227 | longer content → *more* likely flagged declining |
| `sessions_90d` | -0.149 | more sessions → less likely declining |
| `impressions_90d` | -0.120 | more impressions → less likely declining |
| `days_since_last_update` | +0.094 | staler pages → slightly more likely declining |
| `competition` | +0.039 | weak effect |
| `search_volume` | +0.007 | almost no effect |

**Two things stood out to me:**
1. `content_age_days` has a *negative* coefficient — older pages are *less* likely to be flagged as 
   declining in this model. This is counter to my w04 staleness intuition, and lines up with my 
   earlier MIXED verdict on the staleness signal — it's genuinely not a clean predictor here.
2. `days_since_last_update` (my actual "staleness" signal) has only a small positive effect (+0.094) 
   — weak, but at least in the expected direction, unlike raw age.

In [19]:
# Compare predictions vs actual for the test set
test_results = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "predicted_proba": log_proba,
    "predicted_label": (log_proba >= 0.5).astype(int)
})

# False positives: model said "declining" but it wasn't
false_positives = test_results[(test_results["predicted_label"] == 1) & (test_results["actual"] == 0)]

# False negatives: model missed real declining pages
false_negatives = test_results[(test_results["predicted_label"] == 0) & (test_results["actual"] == 1)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))
print("Total test rows:", len(test_results))

from sklearn.metrics import classification_report
print("\n", classification_report(test_results["actual"], test_results["predicted_label"]))

False positives: 761
False negatives: 815
Total test rows: 3614

               precision    recall  f1-score   support

           0       0.51      0.53      0.52      1617
           1       0.61      0.59      0.60      1997

    accuracy                           0.56      3614
   macro avg       0.56      0.56      0.56      3614
weighted avg       0.57      0.56      0.56      3614



**Error analysis:** Out of 3,614 test rows, the model produced **761 false positives** (flagged as 
declining when they weren't) and **815 false negatives** (missed pages that were actually declining). 
Overall accuracy is 0.56 — for the declining class (label 1), precision is 0.61 and recall is 0.59; 
for the non-declining class (label 0), precision is 0.51 and recall is 0.53.

**What this tells me:** the model is only modestly better than a coin flip on raw accuracy (0.56), 
even though its Precision@50 (0.88) is strong — this is an important distinction. Precision@50 only 
measures how good the *top-ranked* predictions are, which is what actually matters for a reviewer 
who only looks at the top 50 candidates. The overall classification numbers (56% accuracy, ~0.60 
F1) show the model is much less confident/accurate on the bulk of "middle" pages — it's good at 
identifying the *most obvious* declining pages, but far less reliable for borderline cases. This 
matches how the tool would actually be used: as a ranking/priority tool for the top of the queue, 
not a general-purpose classifier that should be trusted on every single page.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.